# 01 — Batch lakehouse: bronze / silver / gold (EMR)

Build the medallion pipeline from the batch seed files created in `00`, using the shared `retail_lakehouse` transformation functions — the same functions the streaming path in `03` reuses for silver enrichment. This is the baseline to compare the streaming path against.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

## Read the seed files and dimensions

In [ ]:
from retail_lakehouse.transformations import (
    normalize_click_events, filter_valid_events, deduplicate_events,
    enrich_with_product_customer, revenue_by_hour, add_ingest_metadata,
)
from pyspark.sql import functions as F

raw_events = spark.read.json(cfg.path("source", "events_json"))
products = spark.table(cfg.table("dim_product_seed"))
customers = spark.table(cfg.table("dim_customer_seed"))
print("Raw event rows:", raw_events.count())

## Bronze — minimally transformed, ingestion metadata only

In [ ]:
bronze = add_ingest_metadata(raw_events, "batch_file_seed")
bronze.write.mode("overwrite").format("delta").partitionBy("_ingest_date").saveAsTable(cfg.table("bronze_clickstream_batch"))
spark.table(cfg.table("bronze_clickstream_batch")).limit(5).toPandas()

## Silver — normalized, validated, deduplicated

In [ ]:
silver = deduplicate_events(filter_valid_events(normalize_click_events(bronze)))
silver.write.mode("overwrite").format("delta").partitionBy("event_date").saveAsTable(cfg.table("silver_clickstream_batch"))
print("Silver rows:", spark.table(cfg.table("silver_clickstream_batch")).count())

## Gold — enriched with dimensions, aggregated to a business metric

In [ ]:
enriched = enrich_with_product_customer(spark.table(cfg.table("silver_clickstream_batch")), products, customers)
gold = revenue_by_hour(enriched.withColumn(
    "is_purchase", F.col("event_type").isin("purchase", "checkout")
))
gold.write.mode("overwrite").format("delta").saveAsTable(cfg.table("gold_revenue_by_hour_batch"))

spark.table(cfg.table("gold_revenue_by_hour_batch")).orderBy(F.desc("revenue")).toPandas()

## Query plan check

Confirm the product join broadcasts (small dimension) while the aggregation shuffles (expected for `groupBy`). See `class-emr/01_spark_batch_processing.ipynb` for the full explanation of this plan shape.

In [ ]:
enriched.explain("formatted")

## Next

`02_kafka_msk_streaming_ingest.ipynb` ingests the same kind of event from a live MSK topic instead of a static file, and `03_streaming_silver_gold_delta.ipynb` reuses `normalize_click_events`/`enrich_with_product_customer` again — the transformation code doesn't care whether it's fed by batch or streaming, only the orchestration around it differs.